In [1]:
import datetime
import time
import tarfile

import boto3
import pandas as pd
import numpy as np
# from sagemaker import get_execution_role
import sagemaker
from sklearn.model_selection import train_test_split
from sklearn.datasets import fetch_openml

from sagemaker.core.helper.session_helper import Session, get_execution_role


sm_boto3 = boto3.client("sagemaker")

sess = Session()

role = get_execution_role()

region = sess.boto_session.region_name

bucket = sess.default_bucket()  # this could also be a hard-coded bucket name

print("Using bucket " + bucket)



sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml
Using bucket sagemaker-us-east-1-155361516759


In [2]:
# we use the Boston housing dataset
data = fetch_openml(name='boston', version=1, as_frame=True)

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    data.data, data.target, test_size=0.25, random_state=42
)

trainX = pd.DataFrame(X_train, columns=data.feature_names)
trainX["target"] = y_train

testX = pd.DataFrame(X_test, columns=data.feature_names)
testX["target"] = y_test



In [4]:
trainX.head()

trainX.to_csv("boston_train.csv")
testX.to_csv("boston_test.csv")

# send data to S3. SageMaker will take training data from s3
trainpath = sess.upload_data(
    path="boston_train.csv", bucket=bucket, key_prefix="sagemaker/sklearncontainer"
)

testpath = sess.upload_data(
    path="boston_test.csv", bucket=bucket, key_prefix="sagemaker/sklearncontainer"
)



In [5]:
%%writefile script.py

import argparse
import joblib
import os

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor


# inference functions ---------------
def model_fn(model_dir):
    clf = joblib.load(os.path.join(model_dir, "model.joblib"))
    return clf


if __name__ == "__main__":

    print("extracting arguments")
    parser = argparse.ArgumentParser()

    # hyperparameters sent by the client are passed as command-line arguments to the script.
    # to simplify the demo we don't use all sklearn RandomForest hyperparameters
    parser.add_argument("--n-estimators", type=int, default=10)
    parser.add_argument("--min-samples-leaf", type=int, default=3)

    # Data, model, and output directories
    parser.add_argument("--model-dir", type=str, default=os.environ.get("SM_MODEL_DIR"))
    parser.add_argument("--train", type=str, default=os.environ.get("SM_CHANNEL_TRAIN"))
    parser.add_argument("--test", type=str, default=os.environ.get("SM_CHANNEL_TEST"))
    parser.add_argument("--train-file", type=str, default="boston_train.csv")
    parser.add_argument("--test-file", type=str, default="boston_test.csv")
    parser.add_argument(
        "--features", type=str
    )  # in this script we ask user to explicitly name features
    parser.add_argument(
        "--target", type=str
    )  # in this script we ask user to explicitly name the target

    args, _ = parser.parse_known_args()

    print("reading data")
    train_df = pd.read_csv(os.path.join(args.train, args.train_file))
    test_df = pd.read_csv(os.path.join(args.test, args.test_file))

    print("building training and testing datasets")
    X_train = train_df[args.features.split()]
    X_test = test_df[args.features.split()]
    y_train = train_df[args.target]
    y_test = test_df[args.target]

    # train
    print("training model")
    model = RandomForestRegressor(
        n_estimators=args.n_estimators, min_samples_leaf=args.min_samples_leaf, n_jobs=-1
    )

    model.fit(X_train, y_train)

    # print abs error
    print("validating model")
    abs_err = np.abs(model.predict(X_test) - y_test)

    # print couple perf metrics
    for q in [10, 50, 90]:
        print("AE-at-" + str(q) + "th-percentile: " + str(np.percentile(a=abs_err, q=q)))

    # persist model
    path = os.path.join(args.model_dir, "model.joblib")
    joblib.dump(model, path)
    print("model persisted at " + path)
    print(args.min_samples_leaf)



Writing script.py


In [6]:
# Optional local smoke test only.
# This model.joblib is not used for deployment.
! python script.py --n-estimators 100 \
                   --min-samples-leaf 2 \
                   --model-dir ./ \
                   --train ./ \
                   --test ./ \
                   --features 'CRIM ZN INDUS CHAS NOX RM AGE DIS RAD TAX PTRATIO B LSTAT' \
                   --target target

extracting arguments
reading data
building training and testing datasets
training model
validating model
AE-at-10th-percentile: 0.32061119047618264
AE-at-50th-percentile: 1.519720238095239
AE-at-90th-percentile: 4.465691904761905
model persisted at ./model.joblib
2


In [7]:
# We use the Estimator from the SageMaker Python SDK
# from sagemaker.sklearn.estimator import SKLearn  <- deprecated


# FRAMEWORK_VERSION = "0.23-1"

# sklearn_estimator = SKLearn(  <- deprecated : check code blocks below for updated SageMaker v3 API utilization
#     entry_point="script.py",
#     role=get_execution_role(),
#     instance_count=1,
#     instance_type="ml.m4.xlarge",#"ml.c5.xlarge",#"ml.m4.xlarge",
#     framework_version=FRAMEWORK_VERSION,
#     base_job_name="rf-scikit",
#     metric_definitions=[{"Name": "median-AE", "Regex": "AE-at-50th-percentile: ([0-9.]+).*$"}],
#     hyperparameters={
#         "n-estimators": 100,
#         "min-samples-leaf": 3,
#         "features": "CRIM ZN INDUS CHAS NOX RM AGE DIS RAD TAX PTRATIO B LSTAT",
#         "target": "target",
#     },
# )

# import time
# tic = time.time()

# # launch training job, with asynchronous call
# sklearn_estimator.fit({"train": trainpath, "test": testpath}, wait=False)

# toc = time.time()
# print(toc - tic)


In [8]:
#  TODO: Set up debugging and profiling rules and hooks
# The hooks and rules now live inside train.py; this cell prepares the job
import os
import shutil
import boto3
from sagemaker.core import image_uris


data_s3 = f"s3://{bucket}/sagemaker/sklearncontainer/data"
output_s3 = f"s3://{bucket}/sagemaker/sklearncontainer/model"

# Dedicated source dir: using "." would upload data/dogImages (~1.1 GB) along with the code.
# train.py stays at the project root (used by the local debugging block) and is copied here.
source_dir = "scripts"
os.makedirs(source_dir, exist_ok=True)
shutil.copy("script.py", f"{source_dir}/script.py")

# smdebug is no longer needed; tensorboard + psutil are
with open(f"{source_dir}/requirements.txt", "w") as f:
    f.write("psutil\n")

training_image = image_uris.retrieve(
    framework="sklearn",
    region=region,
    version="1.9-0",
    py_version="py3",
    image_scope="training",
    instance_type="ml.m5.large"
)

print(role)
print(data_s3)
print(output_s3)
print(training_image)

arn:aws:iam::155361516759:role/service-role/AmazonSageMaker-ExecutionRole-20261023T161945
s3://sagemaker-us-east-1-155361516759/sagemaker/sklearncontainer/data
s3://sagemaker-us-east-1-155361516759/sagemaker/sklearncontainer/model
683313688378.dkr.ecr.us-east-1.amazonaws.com/sagemaker-scikit-learn:1.9-0-cpu-py3


In [9]:
!python -m py_compile scripts/script.py && echo "OK"

OK


In [10]:
from sagemaker.core.shapes import MetricDefinition

base_job_name="rf-scikit"

# metric_definitions=[{"Name": "median-AE", "Regex": "AE-at-50th-percentile: ([0-9.]+).*$"}]
metric_definitions = [
    MetricDefinition(
        name="median-AE",
        regex="AE-at-50th-percentile: ([0-9.]+).*$",
    )
]

hyperparameters={
        "n-estimators": 100,
        "min-samples-leaf": 3,
        "features": "CRIM ZN INDUS CHAS NOX RM AGE DIS RAD TAX PTRATIO B LSTAT",
        "target": "target",
    }
print(metric_definitions)

print(hyperparameters)

[MetricDefinition(name='median-AE', regex='AE-at-50th-percentile: ([0-9.]+).*$')]
{'n-estimators': 100, 'min-samples-leaf': 3, 'features': 'CRIM ZN INDUS CHAS NOX RM AGE DIS RAD TAX PTRATIO B LSTAT', 'target': 'target'}


In [11]:
# TODO: Create and fit an estimator
from sagemaker.train import ModelTrainer
from sagemaker.core.training.configs import (
    Compute,
    InputData,
    OutputDataConfig,
    SourceCode,
    StoppingCondition,
)

trainer = ModelTrainer(
    training_image=training_image,
    role=role,
    sagemaker_session=sess,
    base_job_name=base_job_name,
    source_code=SourceCode(
        source_dir=source_dir,
        requirements="requirements.txt",
        entry_script="script.py",
    ),
    compute=Compute(instance_type="ml.m5.large", instance_count=1),
    output_data_config=OutputDataConfig(s3_output_path=output_s3),
    hyperparameters=hyperparameters,
    stopping_condition=StoppingCondition(max_runtime_in_seconds=3600)
).with_metric_definitions(metric_definitions)

[10/31/26 07:59:09] INFO     OutputDataConfig compression type not provided. Using default:         ]8;id=16626390;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=16626391;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/defaults.py#165\165]8;;\
                             GZIP                                                                                  

                    INFO     Training image URI:                                               ]8;id=16626398;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/model_trainer.py\model_trainer.py]8;;\:]8;id=16626399;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/model_trainer.py#558\558]8;;\
                             683313688378.dkr.ecr.us-east-1.amazonaws.com/sagemaker-scikit-lea                     
                             rn:1.9-0-cpu-py3                                                                      

In [12]:
data_input = f"s3://{bucket}/sagemaker/sklearncontainer/"

tic = time.time()

trainer.train(
    input_data_config=[
        InputData(channel_name="train", data_source=data_input),
        InputData(channel_name="test", data_source=data_input),
    ],
    wait=True
) # TODO: Remember to include your data channels

toc = time.time()
print(f"Training time: {(toc - tic)/60} min")

                    INFO     SageMaker Python SDK will collect telemetry to help us better ]8;id=16626406;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=16626407;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/telemetry/telemetry_logging.py#325\325]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#conf                         
                             iguring-and-using-defaults-with-the-sagemaker-python-sdk.                             

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


                    INFO     Creating training_job resource.                                     ]8;id=16626414;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=16626415;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py#31296\31296]8;;\

                    WARNING  No region provided. Using default region.                                 ]8;id=16626422;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/utils/utils.py\utils.py]8;;\:]8;id=16626423;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/utils/utils.py#350\350]8;;\

Output()

...

                    INFO     rf-scikit-20261031075909/algo-1-1793433597:                         ]8;id=16627035;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=16627036;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py#31642\31642]8;;\
                             3                                                                                     

                    INFO     rf-scikit-20261031075909/algo-1-1793433597:                         ]8;id=16627041;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=16627042;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py#31642\31642]8;;\
                             ++ echo 'Training Container Execution Completed'                                      

                    INFO     rf-scikit-20261031075909/algo-1-1793433597:                         ]8;id=16627047;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=16627048;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py#31642\31642]8;;\
                             Training Container Execution Completed                                                

[10/31/26 08:01:27] INFO     Final Resource Status: Completed                                    ]8;id=16627054;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=16627055;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py#31648\31648]8;;\

Training time: 2.309492878119151 min


In [13]:
# Create the inference folder and script
import os

# Create a folder dedicated to endpoint inference code.
inference_dir = "./inference"
os.makedirs(inference_dir, exist_ok=True)

inference_script = """
import joblib
import os


def model_fn(model_dir):
    # SageMaker extracts model.tar.gz into model_dir.
    # model.joblib was saved to SM_MODEL_DIR during training.
    model_path = os.path.join(model_dir, "model.joblib")
    return joblib.load(model_path)
"""

with open(os.path.join(inference_dir, "inference.py"), "w") as file:
    file.write(inference_script)

print("Created:", os.path.join(inference_dir, "inference.py"))

Created: ./inference/inference.py


In [14]:
# Deprecated code block : check code block below for updated deployment code
# sklearn_estimator.latest_training_job.wait(logs="None")
# artifact = sm_boto3.describe_training_job(
#     TrainingJobName=sklearn_estimator.latest_training_job.name
# )["ModelArtifacts"]["S3ModelArtifacts"]

# print("Model artifact persisted at " + artifact)

# from sagemaker.sklearn.model import SKLearnModel

# model = SKLearnModel(
#     model_data=artifact,
#     role=get_execution_role(),
#     entry_point="script.py",
#     framework_version=FRAMEWORK_VERSION,
# )

# predictor = model.deploy(instance_type="ml.c5.large", initial_instance_count=1)



In [15]:
import boto3

from sagemaker.core import image_uris
from sagemaker.core.training.configs import SourceCode
from sagemaker.serve import ModelBuilder
from sagemaker.serve.utils.types import ModelServer


# ---------------------------------------------------------
# 1. Retrieve the model.tar.gz produced by the training job
# ---------------------------------------------------------

training_job_name =  trainer._latest_training_job.training_job_name

sm_client = boto3.client("sagemaker", region_name=region)

training_job_description = sm_client.describe_training_job(
    TrainingJobName=training_job_name
)

model_artifact_s3_uri = training_job_description[
    "ModelArtifacts"
]["S3ModelArtifacts"]

print("Model artifact:", model_artifact_s3_uri)



Model artifact: s3://sagemaker-us-east-1-155361516759/sagemaker/sklearncontainer/model/rf-scikit-20261031075909/output/model.tar.gz


In [16]:
# Repacking
import os
import shutil
import tarfile

s3 = boto3.client("s3")
src_bucket, src_key = model_artifact_s3_uri.replace("s3://", "").split("/", 1)
s3.download_file(src_bucket, src_key, "model.tar.gz")

shutil.rmtree("repack", ignore_errors=True)
os.makedirs("repack/code")
with tarfile.open("model.tar.gz") as t:
    t.extractall("repack", filter="data")                      # -> repack/model.pth
shutil.copy("inference/inference.py", "repack/code/inference.py")  # -> repack/code/inference.py

with tarfile.open("model_with_code.tar.gz", "w:gz") as t:
    for name in os.listdir("repack"):
        t.add(os.path.join("repack", name), arcname=name)

# Sanity check: should list model.pth and code/inference.py
with tarfile.open("model_with_code.tar.gz") as t:
    print(t.getnames())

model_with_code_url = sess.upload_data(
    path="model_with_code.tar.gz", bucket=bucket, key_prefix=f"sagemaker/sklearncontainer/model_deploy"
)
print(f"Repacked model artifact: {model_with_code_url}")

['code', 'code/inference.py', 'model.joblib']
Repacked model artifact: s3://sagemaker-us-east-1-155361516759/sagemaker/sklearncontainer/model_deploy/model_with_code.tar.gz


In [17]:
# sm_client.create_model(
#     ModelName=model_name,
#     ExecutionRoleArn=role,
#     PrimaryContainer={
#         "Image": inference_image_uri,
#         "ModelDataUrl": repacked_model_s3_uri,
#         "Environment": {
#             "SAGEMAKER_PROGRAM": "inference.py",

#             # The container downloads this archive and finds
#             # code/inference.py inside it.
#             "SAGEMAKER_SUBMIT_DIRECTORY": repacked_model_s3_uri,

#             "SAGEMAKER_CONTAINER_LOG_LEVEL": "20",
#             "SAGEMAKER_REGION": region,
#         },
#     },
# )

from sagemaker.serve import ModelBuilder

# predictor=trainer.deploy(initial_instance_count=1, instance_type="ml.m5.large")  # TODO: Add your deployment configuration like instance type and number of instances

inference_image_uri = image_uris.retrieve(
    framework="sklearn",
    region=region,
    version="1.9-0",
    py_version="py3",
    image_scope="inference",
    instance_type="ml.m5.large"
)

builder = ModelBuilder(
    model_path=model_with_code_url,
    image_uri=inference_image_uri,
    role_arn=role,
    env_vars={
        "SAGEMAKER_PROGRAM": "inference.py", 
        "SAGEMAKER_SUBMIT_DIRECTORY": "/opt/ml/model/code"
    },
)
model = builder.build()

model

[10/31/26 08:01:31] DEBUG    Auto-detecting optimal instance type for model...           ]8;id=16627062;file:///opt/conda/lib/python3.12/site-packages/sagemaker/serve/model_builder_utils.py\model_builder_utils.py]8;;\:]8;id=16627063;file:///opt/conda/lib/python3.12/site-packages/sagemaker/serve/model_builder_utils.py#340\340]8;;\

                    DEBUG    Using default CPU instance type: ml.m5.large                ]8;id=16627069;file:///opt/conda/lib/python3.12/site-packages/sagemaker/serve/model_builder_utils.py\model_builder_utils.py]8;;\:]8;id=16627070;file:///opt/conda/lib/python3.12/site-packages/sagemaker/serve/model_builder_utils.py#374\374]8;;\

[10/31/26 08:01:32] DEBUG    No ModelMetadata provided. ModelBuilder is not handling    ]8;id=16627076;file:///opt/conda/lib/python3.12/site-packages/sagemaker/serve/model_builder_utils.py\model_builder_utils.py]8;;\:]8;id=16627077;file:///opt/conda/lib/python3.12/site-packages/sagemaker/serve/model_builder_utils.py#1335\1335]8;;\
                             MLflow model input                                                                    

                    INFO     Creating model with name: model-676dbeea                        ]8;id=16627084;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/helper/session_helper.py\session_helper.py]8;;\:]8;id=16627085;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/helper/session_helper.py#2029\2029]8;;\

                    INFO     ✅ Model has been created: 'model-676dbeea' using server None in ]8;id=16627092;file:///opt/conda/lib/python3.12/site-packages/sagemaker/serve/model_builder.py\model_builder.py]8;;\:]8;id=16627093;file:///opt/conda/lib/python3.12/site-packages/sagemaker/serve/model_builder.py#3357\3357]8;;\
                             SAGEMAKER_ENDPOINT mode (ARN:                                                         
                             arn:aws:sagemaker:us-east-1:155361516759:model/model-676dbeea)                        

Model(model_name='model-676dbeea', primary_container=ContainerDefinition(container_hostname=Unassigned(), image='683313688378.dkr.ecr.us-east-1.amazonaws.com/sagemaker-scikit-learn:1.9-0-cpu-py3', image_config=Unassigned(), mode='SingleModel', model_data_url='s3://sagemaker-us-east-1-155361516759/sagemaker/sklearncontainer/model_deploy/model_with_code.tar.gz', model_data_source=ModelDataSource(s3_data_source=S3ModelDataSource(s3_data_type='S3Object', compression_type='Gzip', s3_uri='s3://sagemaker-us-east-1-155361516759/sagemaker/sklearncontainer/model_deploy/model_with_code.tar.gz', model_access_config=Unassigned(), hub_access_config=Unassigned(), manifest_s3_uri=Unassigned(), e_tag=Unassigned(), manifest_etag=Unassigned())), additional_model_data_sources=Unassigned(), environment={'SAGEMAKER_CONTAINER_LOG_LEVEL': '20', 'SAGEMAKER_PROGRAM': 'inference.py', 'SAGEMAKER_REGION': 'us-east-1', 'SAGEMAKER_SUBMIT_DIRECTORY': '/opt/ml/model/code'}, model_package_name=Unassigned(), inference_s

In [18]:
predictor = builder.deploy(initial_instance_count=1, instance_type="ml.m5.large")

                    INFO     Creating endpoint-config with name endpoint-72d87e24            ]8;id=16627099;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/helper/session_helper.py\session_helper.py]8;;\:]8;id=16627100;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/helper/session_helper.py#1017\1017]8;;\

[10/31/26 08:01:33] INFO     Creating endpoint with name endpoint-72d87e24                   ]8;id=16627106;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/helper/session_helper.py\session_helper.py]8;;\:]8;id=16627107;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/helper/session_helper.py#1232\1232]8;;\

                    WARNING  Failed to enable live logging: An error occurred                ]8;id=16627113;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/helper/session_helper.py\session_helper.py]8;;\:]8;id=16627114;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/helper/session_helper.py#2951\2951]8;;\
                             (AccessDeniedException) when calling the FilterLogEvents                              
                             operation: User:                                                                      
                             arn:aws:sts::155361516759:assumed-role/AmazonSageMaker-Executio                       
                             nRole-20261023T161945/SageMaker is not authorized to perform:                         
                             logs:FilterLogEvents on resource:                                                     
                             arn:aws:logs:us-east-1:155361516759:log-group:/aws/sagemaker/En                       
                             dpoints/endpoint-72d87e24 because no identity-based policy                            
                             allows the logs:FilterLogEvents action. Fallback to default                           
                             logging...                                                                            

Output()

[10/31/26 08:04:34] INFO     ✅ Deployment successful: Endpoint 'endpoint-72d87e24' using     ]8;id=16627120;file:///opt/conda/lib/python3.12/site-packages/sagemaker/serve/model_builder.py\model_builder.py]8;;\:]8;id=16627121;file:///opt/conda/lib/python3.12/site-packages/sagemaker/serve/model_builder.py#2687\2687]8;;\
                             None in SAGEMAKER_ENDPOINT mode (ARN:                                                 
                             arn:aws:sagemaker:us-east-1:155361516759:endpoint/endpoint-72d87                      
                             e24)                                                                                  

In [19]:
import time 

In [20]:
# print(predictor.predict(testX[data.feature_names]))
test_sample = testX[data.feature_names]
# Convert the DataFrame to CSV expected by the sklearn container
payload = test_sample.to_csv(
    index=False,
    header=False,
)

start_time = time.time()

while True:
    elapsed_time = time.time() - start_time

    response = predictor.invoke(
    body=payload,
    content_type="text/csv",
    )

    result = response.body.read().decode("utf-8")
    print(result)

    if elapsed_time >= 45:
        break

print("45 seconds passed!")

[22.73826305916305, 31.796446753246766, 17.093914876789874, 23.66987301587301, 16.737141164391172, 21.572006349206355, 19.126970887445893, 16.00901042568542, 21.28262323232324, 21.06461024531024, 19.90105080752581, 19.64274657287158, 8.276672691197694, 21.4852713924964, 19.52460093517594, 25.616559451659445, 18.401393520368526, 8.70922474747475, 44.16705591630592, 15.41354285714287, 24.14952067099568, 24.048655699855704, 14.857469877344887, 24.103146825396834, 14.772184126984126, 15.376699603174611, 21.488115836940832, 14.033515159840157, 19.729168833943834, 21.047692316017304, 20.12833536463536, 23.528556493506493, 29.00731735209234, 20.42565295815296, 14.594642099567102, 15.970093165168171, 35.23016767676768, 19.289623376623375, 21.16838691586192, 23.918713431013433, 19.33291688311688, 28.968683910533915, 44.219889249639245, 19.561673196248197, 22.616516955266952, 13.266604256854254, 15.405939749139753, 24.385352716727734, 18.238138095238106, 28.692179112554108, 21.36098257575757, 33

In [21]:
#sm_boto3.delete_endpoint(EndpointName=predictor.endpoint_name)
predictor.delete()
print("Endpoint successfully deleted.")

[10/31/26 08:05:19] INFO     Deleting Endpoint - endpoint-72d87e24                               ]8;id=16627127;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=16627128;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py#10427\10427]8;;\

Endpoint successfully deleted.
